In [2]:
!pip install -U ginza ja_ginza

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.0/31.0 MB 82.9 MB/s  0:00:00m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 779.7/779.7 kB 56.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 68.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 82.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 82.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 82.6 MB/s  0:00:006m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22/22 [ja_ginza]/22 [ja_ginza]

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [3]:
import spacy

nlp = spacy.load("ja_ginza")

doc = nlp("私は大学で自然言語処理を研究しています。")

for token in doc:
    print(token.text, token.lemma_, token.pos_)

/usr/local/lib/python3.10/dist-packages/torch/cuda/__init__.py:827: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


私 私 PRON
は は ADP
大学 大学 NOUN
で で ADP
自然言語処理 自然言語処理 PROPN
を を ADP
研究 研究 VERB
し する AUX
て て SCONJ
い いる VERB
ます ます AUX
。 。 PUNCT


In [5]:
import glob
import os
import re
from collections import Counter
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import spacy

# --- GiNZAの初期化 ---
nlp = spacy.load("ja_ginza")

# GiNZAのUniversal Dependencies品詞分類に基づく設定
# ADP: 助詞, AUX: 助動詞, CCONJ/SCONJ: 接続詞, DET: 連体詞, PRON: 代名詞
FUNCTION_POS_TAGS = {"ADP", "AUX", "CCONJ", "SCONJ", "DET", "PRON"}
# NOUN/PROPN: 名詞, VERB: 動詞, ADJ: 形容詞, ADV: 副詞, INTJ: 感動詞
CONTENT_POS_TAGS = {"NOUN", "PROPN", "VERB", "ADJ", "ADV", "INTJ"}

def extract_dependency_pairs(doc):
    """GiNZAの解析済みドキュメントから「元単語_係り先単語」のペアを抽出する"""
    pairs = []
    for token in doc:
        # 記号、空白、句読点は除外
        if token.pos_ in {"PUNCT", "SYM", "SPACE"}:
            continue
        # 自分自身以外への係り先（親トークン）がある場合
        if token.head and token.head != token:
            if token.head.pos_ in {"PUNCT", "SYM", "SPACE"}:
                continue
            # 「元の単語_係り先の単語」を1つのトークンにする
            pairs.append(f"{token.orth_}_{token.head.orth_}")
    return " ".join(pairs)

def analyze_text_with_ginza(folder_path):
    """GiNZAのみを用いて、テキストの語彙・品詞・構文特徴量を一括抽出する"""
    current_dir = os.getcwd()

    # 【ファイルの指定】
    # ▼ パターンA: 1ファイルテスト用（現在はこっち）
    filepaths = [os.path.join(current_dir, "Human_text_01.txt")]
    
    # ▼ パターンB: 全ファイル用（30個一括にする時は下の3行の#を消す）
    # base_folder = os.path.join(current_dir, "Human_text")
    # filepaths = sorted(
    #     glob.glob(os.path.join(base_folder, "*.txt")),
    #     key=lambda x: int(re.search(r"\d+", os.path.basename(x)).group()),
    # )

    rows = []
    dependency_corpus = [] # ベクトル化用のテキストを溜めるリスト

    for filepath in filepaths:
        if not os.path.exists(filepath):
            continue

        with open(filepath, "r", encoding="utf-8") as f:
            text = f.read()

        file_name = os.path.basename(filepath)
        
        # GiNZAで解析を実行
        doc = nlp(text)

        # 各種カウンターの初期化
        word_count = 0
        func_count = 0
        content_count = 0
        lemmas = [] # 原形（TTR用）
        
        # 1. 語彙・品詞の統計計算
        for token in doc:
            if token.pos_ in {"PUNCT", "SYM", "SPACE"}:
                continue
            
            word_count += 1
            lemmas.append(token.lemma_) # 単語の原形（基本形）

            if token.pos_ in FUNCTION_POS_TAGS:
                func_count += 1
            elif token.pos_ in CONTENT_POS_TAGS:
                content_count += 1

        # 指標の計算
        ttr = len(set(lemmas)) / word_count if word_count > 0 else 0
        func_ratio = (func_count / word_count) * 100 if word_count > 0 else 0
        content_ratio = (content_count / word_count) * 100 if word_count > 0 else 0

        # 2. 係り受けペアテキストの抽出
        dep_text = extract_dependency_pairs(doc)
        dependency_corpus.append(dep_text)
        pair_count = len(dep_text.split())

        # 3. データの蓄積
        rows.append({
            "ファイル名": file_name,
            "総単語数(GiNZA)": word_count,
            "異なり語数(原形)": len(set(lemmas)),
            "TTR(語彙多様性)": round(ttr, 4),
            "機能語割合(%)": round(func_ratio, 2),
            "内容語割合(%)": round(content_ratio, 2),
            "総係り受けペア数": pair_count,
            "_dep_text_internal": dep_text # 内部ベクトル化用
        })

    df = pd.DataFrame(rows)
    return df, dependency_corpus

# --- メイン実行セクション ---
df_res, corpus = analyze_text_with_ginza("Human_text")

if not df_res.empty:
    # 係り受けペアのベクトル化
    vectorizer = CountVectorizer(max_features=1000)
    X_dep = vectorizer.fit_transform(corpus)
    feature_names = vectorizer.get_feature_names_out()

    print("=== GiNZA 詳細解析結果出力 ===")
    
    # ループを回して、1ファイルごとにご希望の全統計数値を綺麗にプリント出力します
    for i, row in df_res.iterrows():
        print(f"\n=== ファイル：{row['ファイル名']} ===")
        print(f"総単語数：{row['総単語数(GiNZA)']}")
        print(f"異なり語数：{row['異なり語数(原形)']}")
        print(f"TTR：{row['TTR(語彙多様性)']:.4f}")
        print(f"機能語割合(%)：{row['機能語割合(%)']:.2f}")
        print(f"内容語割合(%)：{row['内容語割合(%)']:.2f}")
        print(f"総係り受けペア数：{row['総係り受けペア数']}")
        
        # ─── ここから下で、ご希望の細かい指標をドキュメントから再集計して表示 ───
        # ※ 個別表示のために再解析して詳細を出力します
        with open(os.path.join(os.getcwd(), "Human_text", row['ファイル名']) if "Human_text" in os.listdir(".") else os.path.join(os.getcwd(), row['ファイル名']), "r", encoding="utf-8") as f:
            raw_text = f.read()
        
        doc = nlp(raw_text)
        
        # 文の集計
        sentences = [s.text.strip() for s in doc.sents if s.text.strip()]
        avg_sent_len = np.mean([len(s) for s in sentences]) if sentences else 0
        print(f"平均文長：{avg_sent_len:.4f}")
        
        # 品詞の詳細割合 (GiNZAのタグを日本語に翻訳してカウント)
        pos_counts = Counter()
        particle_counts = Counter()
        total_valid_words = 0
        
        for token in doc:
            if token.pos_ in {"PUNCT", "SYM", "SPACE"}:
                continue
            total_valid_words += 1
            
            # Universal Dependencies から日本語の主要品詞へのマッピング
            if token.pos_ in {"NOUN", "PROPN"}:
                pos_counts["名詞"] += 1
            elif token.pos_ == "VERB":
                pos_counts["動詞"] += 1
            elif token.pos_ == "ADJ":
                pos_counts["形容詞"] += 1
            elif token.pos_ == "ADP":
                pos_counts["助詞"] += 1
                particle_counts[token.orth_] += 1 # 助詞の個別カウント
            elif token.pos_ == "AUX":
                pos_counts["助動詞"] += 1
            elif token.pos_ == "ADV":
                pos_counts["副詞"] += 1

        # 各品詞割合の出力
        for pos_name in ["名詞", "動詞", "形容詞", "助詞", "助動詞", "副詞"]:
            ratio = (pos_counts[pos_name] / total_valid_words) * 100 if total_valid_words > 0 else 0
            print(f"{pos_name}割合：{ratio:.4f}")
            
        # 助詞の個別内訳割合の出力
        total_particles = sum(particle_counts.values())
        for p_name, p_count in particle_counts.items():
            p_ratio = p_count / total_particles if total_particles > 0 else 0
            print(f"  助詞_{p_name}：{p_ratio:.4f}")
            
        # 記号類の頻度（単語数に対する割合）
        comma_count = raw_text.count("、")
        period_count = raw_text.count("。")
        print(f"読点頻度：{comma_count / total_valid_words:.4f}" if total_valid_words > 0 else "読点頻度：0.0000")
        print(f"句点頻度：{period_count / total_valid_words:.4f}" if total_valid_words > 0 else "句点頻度：0.0000")
        print(f"感嘆符数：{raw_text.count('！')}")
        print(f"疑問符数：{raw_text.count('？')}")

        # ─── 構文解析（係り受け）のトップ5 ───
        print("\n■ 頻出する係り受けペア（トップ5）:")
        counts = X_dep.toarray()[i]
        top_indices = np.argsort(counts)[::-1][:5]
        for idx in top_indices:
            if counts[idx] > 0:
                print(f"  {feature_names[idx]}：{int(counts[idx])}回")
        print("-" * 40)
        
else:
    print("処理対象のファイルがありません。")

=== GiNZA 詳細解析結果出力 ===

=== ファイル：Human_text_01.txt ===
総単語数：232
異なり語数：84
TTR：0.3621
機能語割合(%)：53.02
内容語割合(%)：46.98
総係り受けペア数：222
平均文長：47.0000
名詞割合：29.3103
動詞割合：11.6379
形容詞割合：5.6034
助詞割合：28.4483
助動詞割合：17.6724
副詞割合：0.4310
  助詞_は：0.1970
  助詞_に：0.0909
  助詞_と：0.0909
  助詞_か：0.0152
  助詞_が：0.1515
  助詞_で：0.1364
  助詞_や：0.0303
  助詞_を：0.1212
  助詞_の：0.1667
読点頻度：0.0690
句点頻度：0.0431
感嘆符数：0
疑問符数：0

■ 頻出する係り受けペア（トップ5）:
  い_て：6回
  し_と：4回
  て_と：4回
  は_スマートフォン：4回
  な_好き：3回
----------------------------------------
